# Phase 07B: Portable Model Factory & FP32 Export

Phase 07B đóng gói model factory của ba Student đã khóa, nạp đúng checkpoint
được Phase 07A lựa chọn và tạo ứng viên triển khai FP32.

Các nguyên tắc:

- chỉ sử dụng T1-S1 seed 123, T2-S2 seed 2026 và T3-S3 seed 42;
- không huấn luyện, không thay checkpoint và không chọn lại theo kết quả export;
- dựng backbone với `pretrained=False`, sau đó nạp toàn bộ trọng số bằng
  `strict=True`; vì vậy notebook không cần Internet;
- xuất encoder và decoder riêng với batch 1, ảnh 224×224, decoder buffer dài 31;
- TorchScript là fallback bắt buộc; ONNX opset 17 là định dạng ưu tiên;
- chỉ dùng input tổng hợp để kiểm tra kỹ thuật, không đọc Validation hay Test;
- lượng tử hóa chưa được phép tại Phase 07B.

## Kaggle inputs bắt buộc

1. Phase 07A Mobile Deployment Entry.
2. Phase 02A Shared Student Pipeline.
3. Phase 04D T1-S1 Sequence-KD seed 123.
4. Phase 04L T2-S2 Sequence-KD seed 2026.
5. Phase 04Q T3-S3 Logit-KD seed 42.

## Runtime

- Accelerator: `None / CPU`.
- Internet: `Off`.
- Không gắn MS COCO.


## Cell 1 — Authorization, manifest và checkpoint integrity


In [1]:
import csv, gc, hashlib, importlib.util, json, os, platform, time
from datetime import datetime, timezone
from pathlib import Path

STARTED = time.perf_counter()
INPUT_ROOT = Path(os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input"))
NOTEBOOK_ROOT = INPUT_ROOT / "notebooks" if (INPUT_ROOT / "notebooks").is_dir() else INPUT_ROOT
OUT = Path(os.environ.get(
    "KLCN_OUTPUT_ROOT",
    "/kaggle/working//phase7_portable_model_factory_export",
))
AUDIT = OUT / "audits"
CONFIG = OUT / "configs"
RESULT = OUT / "results"
EXPORT = OUT / "exports"
PACKAGE = OUT / "package"
MANIFEST = OUT / "manifests"
for directory in [OUT, AUDIT, CONFIG, RESULT, EXPORT, PACKAGE, MANIFEST]:
    directory.mkdir(parents=True, exist_ok=True)

def now():
    return datetime.now(timezone.utc).isoformat()

def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)

def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    temporary.replace(path)

def sha256(path, block_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            block = handle.read(block_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()

def exactly_one(name, root=NOTEBOOK_ROOT):
    matches = sorted(Path(root).rglob(name))
    assert len(matches) == 1, (
        f"Cần đúng một {name}; tìm thấy {len(matches)}: {matches}"
    )
    return matches[0].resolve()

def resolve_inside(root, relative_path):
    root = Path(root).resolve()
    target = (root / relative_path).resolve()
    assert target == root or root in target.parents
    return target

def verify_manifest(marker_path, marker):
    reference = marker["manifest"]["csv"]
    path = resolve_inside(marker_path.parent, reference["relative_path"])
    assert path.is_file()
    assert sha256(path) == reference["sha256"]
    with path.open("r", encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle))
    assert len(rows) == int(marker["manifest_rows"])
    for row in rows:
        target = resolve_inside(marker_path.parent, row["relative_path"])
        assert target.is_file(), target
        assert target.stat().st_size == int(row["size_bytes"])
        assert sha256(target) == row["sha256"]
    return rows

P7A_PATH = exactly_one("PHASE7A_MOBILE_DEPLOYMENT_ENTRY_COMPLETE.json")
AUTH_PATH = exactly_one("PHASE7B_PORTABLE_MODEL_FACTORY_EXPORT_AUTHORIZED.json")
P2_PATH = exactly_one("PHASE2_SHARED_TRAINING_BUNDLE_READY.json")
P7A = read_json(P7A_PATH)
AUTH = read_json(AUTH_PATH)
P2 = read_json(P2_PATH)

assert P7A["status"] == "complete"
assert P7A["decision"] == "phase7a_passed_phase7b_authorized"
assert AUTH["status"] == "authorized"
assert P2["status"] == "ready"
assert AUTH["phase7b_constraints"]["new_training"] is False
assert AUTH["phase7b_constraints"]["test_opening"] is False
assert AUTH["test_set_status"] == "sealed"
assert AUTH["test_set_used"] is False
phase7a_manifest_rows = verify_manifest(P7A_PATH, P7A)

checkpoint_rows = []
for selected in AUTH["selected_checkpoints"]:
    path = exactly_one(selected["checkpoint_file"])
    assert path.stat().st_size == int(selected["checkpoint_size_bytes"])
    assert sha256(path) == selected["checkpoint_sha256"]
    checkpoint_rows.append({**selected, "resolved_path": str(path)})
assert [row["pair"] for row in checkpoint_rows] == ["T1-S1", "T2-S2", "T3-S3"]

entry = {
    "schema_version": "phase7b_cell01_entry_v1",
    "created_at_utc": now(),
    "status": "passed",
    "phase7a_manifest_rows_verified": len(phase7a_manifest_rows),
    "checkpoints": checkpoint_rows,
    "runtime": {
        "python": platform.python_version(),
        "torch_available": importlib.util.find_spec("torch") is not None,
        "torchvision_available": importlib.util.find_spec("torchvision") is not None,
        "timm_available": importlib.util.find_spec("timm") is not None,
        "onnx_available": importlib.util.find_spec("onnx") is not None,
        "onnxruntime_available": importlib.util.find_spec("onnxruntime") is not None,
    },
    "new_training_started": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
}
assert entry["runtime"]["torch_available"]
assert entry["runtime"]["torchvision_available"]
assert entry["runtime"]["timm_available"]
write_json(AUDIT / "phase7b_cell01_entry.json", entry)

print("PHASE 07B CELL 1 — ENTRY")
print("Status                  : PASSED")
print("Phase 07A manifest      :", len(phase7a_manifest_rows), "/", len(phase7a_manifest_rows), "VERIFIED")
print("Selected checkpoints    : 3 / 3 VERIFIED")
print("New training            : NOT STARTED")
print("Held-out Test           : SEALED")
print("Progress                : 1 / 5 code cells")


PHASE 07B CELL 1 — ENTRY
Status                  : PASSED
Phase 07A manifest      : 10 / 10 VERIFIED
Selected checkpoints    : 3 / 3 VERIFIED
New training            : NOT STARTED
Held-out Test           : SEALED
Progress                : 1 / 5 code cells


## Cell 2 — Đóng gói portable model factory


In [2]:
STARTED = time.perf_counter()
FACTORY_SOURCE = 'import math\n\nimport timm\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torchvision.models import mobilenet_v3_large\n\n\ndef select_tensor_from_feature_output(feature_output):\n    if torch.is_tensor(feature_output):\n        return feature_output\n    if isinstance(feature_output, dict):\n        tensors = [value for value in feature_output.values() if torch.is_tensor(value)]\n        if tensors:\n            return tensors[-1]\n    if isinstance(feature_output, (tuple, list)):\n        tensors = [value for value in feature_output if torch.is_tensor(value)]\n        if tensors:\n            return tensors[-1]\n    raise TypeError(f"Unsupported backbone output: {type(feature_output).__name__}")\n\n\ndef feature_tensor_to_tokens(feature_tensor, layout):\n    if layout == "BCHW":\n        tokens = feature_tensor.flatten(2).transpose(1, 2)\n    elif layout == "BHWC":\n        tokens = feature_tensor.flatten(1, 2)\n    elif layout == "BNC":\n        tokens = feature_tensor\n    elif layout == "BCN":\n        tokens = feature_tensor.transpose(1, 2)\n    elif layout == "BC":\n        tokens = feature_tensor.unsqueeze(1)\n    else:\n        raise ValueError(f"Unsupported feature layout: {layout}")\n    return tokens.contiguous()\n\n\nclass StudentVisualEncoderAdapter(nn.Module):\n    def __init__(\n        self,\n        student_id,\n        student_name,\n        backbone,\n        extraction_method,\n        feature_layout,\n        source_feature_dimension,\n        visual_token_count,\n        visual_memory_dimension=256,\n    ):\n        super().__init__()\n        self.student_id = student_id\n        self.student_name = student_name\n        self.backbone = backbone\n        self.extraction_method = extraction_method\n        self.feature_layout = feature_layout\n        self.source_feature_dimension = int(source_feature_dimension)\n        self.visual_token_count = int(visual_token_count)\n        self.visual_memory_dimension = int(visual_memory_dimension)\n        self.projection = nn.Linear(\n            self.source_feature_dimension,\n            self.visual_memory_dimension,\n        )\n        self.visual_position_embedding = nn.Parameter(\n            torch.empty(1, self.visual_token_count, self.visual_memory_dimension)\n        )\n        self.output_norm = nn.LayerNorm(self.visual_memory_dimension)\n        nn.init.xavier_uniform_(self.projection.weight)\n        nn.init.zeros_(self.projection.bias)\n        nn.init.normal_(self.visual_position_embedding, mean=0.0, std=0.02)\n\n    def extract_features(self, images):\n        if self.extraction_method == "torchvision_features":\n            output = self.backbone(images)\n        elif self.extraction_method == "timm_forward_features":\n            output = self.backbone.forward_features(images)\n        else:\n            raise ValueError(self.extraction_method)\n        return select_tensor_from_feature_output(output)\n\n    def forward(self, images):\n        feature_tensor = self.extract_features(images)\n        tokens = feature_tensor_to_tokens(feature_tensor, self.feature_layout)\n        visual_memory = self.projection(tokens)\n        visual_memory = visual_memory + self.visual_position_embedding\n        return self.output_norm(visual_memory)\n\n\nclass SharedCaptionTransformerDecoder(nn.Module):\n    def __init__(self, config):\n        super().__init__()\n        self.config = dict(config)\n        self.vocabulary_size = int(config["vocabulary_size"])\n        self.pad_id = int(config["pad_id"])\n        self.d_model = int(config["d_model"])\n        self.maximum_decoder_steps = int(config["maximum_decoder_steps"])\n        self.token_embedding = nn.Embedding(\n            self.vocabulary_size,\n            self.d_model,\n            padding_idx=self.pad_id,\n        )\n        self.position_embedding = nn.Embedding(\n            self.maximum_decoder_steps,\n            self.d_model,\n        )\n        self.embedding_dropout = nn.Dropout(float(config["dropout"]))\n        layer = nn.TransformerDecoderLayer(\n            d_model=self.d_model,\n            nhead=int(config["nhead"]),\n            dim_feedforward=int(config["dim_feedforward"]),\n            dropout=float(config["dropout"]),\n            activation=config["activation"],\n            batch_first=bool(config["batch_first"]),\n            norm_first=bool(config["norm_first"]),\n            bias=True,\n        )\n        self.transformer_decoder = nn.TransformerDecoder(\n            decoder_layer=layer,\n            num_layers=int(config["num_layers"]),\n            norm=None,\n        )\n        self.final_norm = nn.LayerNorm(self.d_model)\n        self.output_bias = nn.Parameter(torch.zeros(self.vocabulary_size))\n        self.reset_parameters()\n\n    def reset_parameters(self):\n        nn.init.normal_(\n            self.token_embedding.weight,\n            mean=0.0,\n            std=self.d_model ** -0.5,\n        )\n        nn.init.normal_(self.position_embedding.weight, mean=0.0, std=0.02)\n        with torch.no_grad():\n            self.token_embedding.weight[self.pad_id].zero_()\n            self.output_bias.zero_()\n\n    def forward(self, decoder_input_ids, visual_memory, decoder_attention_mask):\n        batch_size, sequence_length = decoder_input_ids.shape\n        positions = torch.arange(\n            sequence_length,\n            device=decoder_input_ids.device,\n            dtype=torch.long,\n        ).unsqueeze(0).expand(batch_size, -1)\n        target = (\n            self.token_embedding(decoder_input_ids) * math.sqrt(self.d_model)\n            + self.position_embedding(positions)\n        )\n        target = self.embedding_dropout(target)\n        causal_mask = torch.triu(\n            torch.ones(\n                sequence_length,\n                sequence_length,\n                dtype=torch.bool,\n                device=decoder_input_ids.device,\n            ),\n            diagonal=1,\n        )\n        hidden = self.transformer_decoder(\n            tgt=target,\n            memory=visual_memory,\n            tgt_mask=causal_mask,\n            tgt_key_padding_mask=~decoder_attention_mask.bool(),\n        )\n        hidden = self.final_norm(hidden)\n        return F.linear(hidden, self.token_embedding.weight, self.output_bias)\n\n\ndef _remove_timm_heads(backbone):\n    if hasattr(backbone, "reset_classifier"):\n        backbone.reset_classifier(num_classes=0, global_pool="")\n    for name in ["head", "head_dist", "classifier", "fc"]:\n        module = getattr(backbone, name, None)\n        if isinstance(module, nn.Module) and not isinstance(module, nn.Identity):\n            setattr(backbone, name, nn.Identity())\n    return backbone\n\n\ndef build_encoder(student_id, contract):\n    if student_id == "S1":\n        backbone = mobilenet_v3_large(weights=None).features\n        extraction_method = "torchvision_features"\n    else:\n        backbone = timm.create_model(contract["model_name"], pretrained=False)\n        backbone = _remove_timm_heads(backbone)\n        extraction_method = "timm_forward_features"\n    feature = contract["feature_extraction"]\n    output = contract["output_contract"]\n    return StudentVisualEncoderAdapter(\n        student_id=student_id,\n        student_name=contract["student_name"],\n        backbone=backbone,\n        extraction_method=extraction_method,\n        feature_layout=feature["raw_layout"],\n        source_feature_dimension=int(feature["source_feature_dimension"]),\n        visual_token_count=int(output["shape"][1]),\n        visual_memory_dimension=int(output["shape"][2]),\n    )\n\n\ndef build_decoder(contract):\n    return SharedCaptionTransformerDecoder(contract["architecture"])\n\n\ndef count_unique_parameters(model):\n    seen = set()\n    total = 0\n    trainable = 0\n    for parameter in model.parameters():\n        identity = id(parameter)\n        if identity in seen:\n            continue\n        seen.add(identity)\n        total += parameter.numel()\n        if parameter.requires_grad:\n            trainable += parameter.numel()\n    return int(total), int(trainable)'
compile(FACTORY_SOURCE, "portable_model_factory.py", "exec")
FACTORY_PATH = PACKAGE / "portable_model_factory.py"
FACTORY_PATH.write_text(FACTORY_SOURCE + "\n", encoding="utf-8")

spec = importlib.util.spec_from_file_location("portable_model_factory", FACTORY_PATH)
factory = importlib.util.module_from_spec(spec)
spec.loader.exec_module(factory)

P2_ROOT = P2_PATH.parent
contract_paths = {
    "decoder": P2_ROOT / "configs/decoder/shared_transformer_decoder_contract.json",
    "S1": P2_ROOT / "configs/encoders/s1_encoder_adapter_contract.json",
    "S2": P2_ROOT / "configs/encoders/s2_encoder_adapter_contract.json",
    "S3": P2_ROOT / "configs/encoders/s3_encoder_adapter_contract.json",
}
for path in contract_paths.values():
    assert path.is_file(), path
contracts = {name: read_json(path) for name, path in contract_paths.items()}

factory_audit = {
    "schema_version": "phase7b_cell02_factory_v1",
    "created_at_utc": now(),
    "status": "passed",
    "factory_file": {
        "relative_path": FACTORY_PATH.relative_to(OUT).as_posix(),
        "size_bytes": FACTORY_PATH.stat().st_size,
        "sha256": sha256(FACTORY_PATH),
    },
    "students": ["S1", "S2", "S3"],
    "backbone_pretrained_download": False,
    "checkpoint_strict_load_required": True,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
}
write_json(AUDIT / "phase7b_cell02_factory.json", factory_audit)

print("PHASE 07B CELL 2 — PORTABLE FACTORY")
print("Status                  : PASSED")
print("Factory SHA-256         :", factory_audit["factory_file"]["sha256"])
print("Pretrained download     : DISABLED")
print("Student factories       : S1 / S2 / S3")
print("Progress                : 2 / 5 code cells")


PHASE 07B CELL 2 — PORTABLE FACTORY
Status                  : PASSED
Factory SHA-256         : 5e0c19926aac7625ad8a29f4bccf68ba685604a9fe4b3b8fad1fab040a56cfe4
Pretrained download     : DISABLED
Student factories       : S1 / S2 / S3
Progress                : 2 / 5 code cells


## Cell 3 — Strict checkpoint load và exact parameter inventory


In [3]:
STARTED = time.perf_counter()
import torch

torch.set_grad_enabled(False)
torch.manual_seed(42)
PAIR_TO_STUDENT = {"T1-S1": "S1", "T2-S2": "S2", "T3-S3": "S3"}

def load_verified_checkpoint(row):
    path = Path(row["resolved_path"])
    assert sha256(path) == row["checkpoint_sha256"]
    try:
        return torch.load(path, map_location="cpu", weights_only=True), "weights_only"
    except Exception as error:
        payload = torch.load(path, map_location="cpu", weights_only=False)
        return payload, f"trusted_pickle_after_sha256:{type(error).__name__}"

models = {}
parameter_rows = []
for row in checkpoint_rows:
    pair = row["pair"]
    student_id = PAIR_TO_STUDENT[pair]
    encoder = factory.build_encoder(student_id, contracts[student_id]).cpu().eval()
    decoder = factory.build_decoder(contracts["decoder"]).cpu().eval()
    payload, load_mode = load_verified_checkpoint(row)
    assert "student_encoder_state" in payload
    assert "student_decoder_state" in payload
    encoder_result = encoder.load_state_dict(payload["student_encoder_state"], strict=True)
    decoder_result = decoder.load_state_dict(payload["student_decoder_state"], strict=True)
    assert not encoder_result.missing_keys and not encoder_result.unexpected_keys
    assert not decoder_result.missing_keys and not decoder_result.unexpected_keys

    encoder_total, encoder_trainable = factory.count_unique_parameters(encoder)
    decoder_total, decoder_trainable = factory.count_unique_parameters(decoder)
    expected_encoder = int(contracts[student_id]["parameters"]["total"])
    expected_decoder = int(
        contracts["decoder"]["parameter_budget"]["actual_total_parameters"]
    )
    assert encoder_total == expected_encoder, (pair, encoder_total, expected_encoder)
    assert decoder_total == expected_decoder, (pair, decoder_total, expected_decoder)

    models[pair] = {"encoder": encoder, "decoder": decoder}
    parameter_rows.append({
        "pair": pair,
        "student_id": student_id,
        "seed": int(row["seed"]),
        "load_mode": load_mode,
        "strict_encoder_load": True,
        "strict_decoder_load": True,
        "encoder_parameters": encoder_total,
        "decoder_parameters": decoder_total,
        "total_parameters": encoder_total + decoder_total,
        "trainable_parameters_before_freeze": encoder_trainable + decoder_trainable,
        "estimated_fp32_parameter_mib": (encoder_total + decoder_total) * 4 / 2**20,
        "estimated_fp16_parameter_mib": (encoder_total + decoder_total) * 2 / 2**20,
    })
    del payload

write_json(RESULT / "phase7b_exact_parameter_inventory.json", {
    "schema_version": "phase7b_exact_parameter_inventory_v1",
    "created_at_utc": now(),
    "rows": parameter_rows,
})
with (RESULT / "phase7b_exact_parameter_inventory.csv").open(
    "w", encoding="utf-8", newline=""
) as handle:
    writer = csv.DictWriter(handle, fieldnames=list(parameter_rows[0].keys()))
    writer.writeheader()
    writer.writerows(parameter_rows)

write_json(AUDIT / "phase7b_cell03_strict_load.json", {
    "schema_version": "phase7b_cell03_strict_load_v1",
    "created_at_utc": now(),
    "status": "passed",
    "rows": parameter_rows,
    "all_state_dict_loads_strict": True,
    "parameter_contracts_exact": True,
    "new_training_started": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})

print("PHASE 07B CELL 3 — STRICT LOAD")
print("Status                  : PASSED")
for row in parameter_rows:
    print(
        f"{row['pair']:7s}                 : "
        f"{row['total_parameters']:,} parameters | "
        f"{row['estimated_fp32_parameter_mib']:.2f} MiB FP32"
    )
print("Checkpoint loads        : STRICT 3 / 3")
print("Progress                : 3 / 5 code cells")


PHASE 07B CELL 3 — STRICT LOAD
Status                  : PASSED
T1-S1                   : 8,990,352 parameters | 34.30 MiB FP32
T2-S2                   : 10,874,112 parameters | 41.48 MiB FP32
T3-S3                   : 10,925,684 parameters | 41.68 MiB FP32
Checkpoint loads        : STRICT 3 / 3
Progress                : 3 / 5 code cells


## Cell 4 — FP32 TorchScript/ONNX export và synthetic parity


In [4]:
STARTED = time.perf_counter()
import numpy as np

HAS_ONNX = importlib.util.find_spec("onnx") is not None
HAS_ORT = importlib.util.find_spec("onnxruntime") is not None
# ONNX decoder legacy export đã vượt giới hạn 12 giờ trên Kaggle.
# Phase 07B khóa TorchScript fallback; ONNX được tách sang nhánh tối ưu riêng.
ATTEMPT_ONNX = False
if HAS_ONNX:
    import onnx
if HAS_ORT:
    import onnxruntime as ort

def max_abs_error(left, right):
    return float(torch.max(torch.abs(left.float() - right.float())).item())

# Khớp protocol FP32 đã khóa tại Phase 07A.
TORCHSCRIPT_ENCODER_ATOL = 1e-4
TORCHSCRIPT_DECODER_LOGIT_ATOL = 1e-4

export_rows = []
for pair, bundle in models.items():
    print(f"Exporting {pair}: TorchScript encoder/decoder...")
    pair_dir = EXPORT / pair.lower().replace("-", "_")
    pair_dir.mkdir(parents=True, exist_ok=True)
    encoder = bundle["encoder"].eval()
    decoder = bundle["decoder"].eval()

    image = torch.linspace(
        -1.0, 1.0, steps=1 * 3 * 224 * 224, dtype=torch.float32
    ).reshape(1, 3, 224, 224)
    token_ids = torch.zeros((1, 31), dtype=torch.long)
    token_ids[:, 0] = 1
    token_ids[:, 1:8] = torch.tensor([5, 11, 23, 47, 89, 144, 233])
    token_mask = torch.zeros((1, 31), dtype=torch.bool)
    token_mask[:, :8] = True

    with torch.inference_mode():
        reference_memory = encoder(image)
        reference_logits = decoder(token_ids, reference_memory, token_mask)
    assert tuple(reference_memory.shape) == (1, 49, 256)
    assert tuple(reference_logits.shape) == (1, 31, 10080)
    assert torch.isfinite(reference_memory).all()
    assert torch.isfinite(reference_logits).all()

    encoder_ts_path = pair_dir / "encoder_fp32.ts"
    decoder_ts_path = pair_dir / "decoder_fp32.ts"
    encoder_ts = torch.jit.trace(encoder, (image,), strict=True)
    encoder_ts = torch.jit.freeze(encoder_ts.eval())
    decoder_ts = torch.jit.trace(
        decoder,
        (token_ids, reference_memory, token_mask),
        strict=True,
    )
    decoder_ts = torch.jit.freeze(decoder_ts.eval())
    torch.jit.save(encoder_ts, str(encoder_ts_path))
    torch.jit.save(decoder_ts, str(decoder_ts_path))
    with torch.inference_mode():
        ts_memory = encoder_ts(image)
        ts_logits = decoder_ts(token_ids, ts_memory, token_mask)
    ts_encoder_error = max_abs_error(reference_memory, ts_memory)
    ts_decoder_error = max_abs_error(reference_logits, ts_logits)
    assert ts_encoder_error <= TORCHSCRIPT_ENCODER_ATOL, (pair, ts_encoder_error)
    assert ts_decoder_error <= TORCHSCRIPT_DECODER_LOGIT_ATOL, (pair, ts_decoder_error)

    encoder_onnx_path = pair_dir / "encoder_fp32.onnx"
    decoder_onnx_path = pair_dir / "decoder_fp32.onnx"
    onnx_status = "deferred_legacy_export_timeout_torchscript_fallback"
    onnx_error = (
        "Legacy ONNX decoder export exceeded Kaggle 12-hour limit; "
        "disabled in Phase 07B to preserve the registered TorchScript fallback."
    )
    ort_encoder_error = None
    ort_decoder_error = None
    if HAS_ONNX and ATTEMPT_ONNX:
        try:
            torch.onnx.export(
                encoder,
                (image,),
                str(encoder_onnx_path),
                export_params=True,
                opset_version=17,
                do_constant_folding=True,
                input_names=["image"],
                output_names=["visual_memory"],
                dynamo=False,
            )
            torch.onnx.export(
                decoder,
                (token_ids, reference_memory, token_mask),
                str(decoder_onnx_path),
                export_params=True,
                opset_version=17,
                do_constant_folding=True,
                input_names=[
                    "decoder_input_ids",
                    "visual_memory",
                    "decoder_attention_mask",
                ],
                output_names=["logits"],
                dynamo=False,
            )
            onnx.checker.check_model(onnx.load(str(encoder_onnx_path)))
            onnx.checker.check_model(onnx.load(str(decoder_onnx_path)))
            onnx_status = "exported_and_checked"
            if HAS_ORT:
                providers = ["CPUExecutionProvider"]
                encoder_session = ort.InferenceSession(
                    str(encoder_onnx_path), providers=providers
                )
                decoder_session = ort.InferenceSession(
                    str(decoder_onnx_path), providers=providers
                )
                ort_memory_np = encoder_session.run(
                    None, {"image": image.numpy()}
                )[0]
                ort_logits_np = decoder_session.run(
                    None,
                    {
                        "decoder_input_ids": token_ids.numpy(),
                        "visual_memory": ort_memory_np,
                        "decoder_attention_mask": token_mask.numpy(),
                    },
                )[0]
                ort_memory = torch.from_numpy(np.asarray(ort_memory_np))
                ort_logits = torch.from_numpy(np.asarray(ort_logits_np))
                ort_encoder_error = max_abs_error(reference_memory, ort_memory)
                ort_decoder_error = max_abs_error(reference_logits, ort_logits)
                onnx_status = "exported_checked_and_ort_parity_measured"
        except Exception as error:
            onnx_status = "failed_torchscript_fallback_retained"
            onnx_error = f"{type(error).__name__}: {error}"

    artifacts = []
    for format_name, path in [
        ("torchscript_encoder", encoder_ts_path),
        ("torchscript_decoder", decoder_ts_path),
        ("onnx_encoder", encoder_onnx_path),
        ("onnx_decoder", decoder_onnx_path),
    ]:
        if path.is_file():
            artifacts.append({
                "format": format_name,
                "relative_path": path.relative_to(OUT).as_posix(),
                "size_bytes": path.stat().st_size,
                "sha256": sha256(path),
            })

    export_rows.append({
        "pair": pair,
        "torchscript_status": "passed",
        "torchscript_encoder_max_abs_error": ts_encoder_error,
        "torchscript_decoder_max_abs_error": ts_decoder_error,
        "onnx_status": onnx_status,
        "onnxruntime_encoder_max_abs_error": ort_encoder_error,
        "onnxruntime_decoder_max_abs_error": ort_decoder_error,
        "onnx_error": onnx_error,
        "artifacts": artifacts,
    })
    print(
        f"{pair} export complete: TorchScript PASSED; "
        f"ONNX {onnx_status}"
    )

write_json(RESULT / "phase7b_export_registry.json", {
    "schema_version": "phase7b_export_registry_v1",
    "created_at_utc": now(),
    "precision": "FP32",
    "input_contract": {
        "image": [1, 3, 224, 224],
        "visual_memory": [1, 49, 256],
        "decoder_input_ids": [1, 31],
        "decoder_attention_mask": [1, 31],
        "logits": [1, 31, 10080],
    },
    "rows": export_rows,
})

all_torchscript_passed = all(
    row["torchscript_status"] == "passed"
    and row["torchscript_encoder_max_abs_error"] <= TORCHSCRIPT_ENCODER_ATOL
    and row["torchscript_decoder_max_abs_error"] <= TORCHSCRIPT_DECODER_LOGIT_ATOL
    for row in export_rows
)
assert all_torchscript_passed
write_json(AUDIT / "phase7b_cell04_export.json", {
    "schema_version": "phase7b_cell04_export_v1",
    "created_at_utc": now(),
    "status": "passed",
    "torchscript_fallback_passed_3_of_3": all_torchscript_passed,
    "registered_fp32_thresholds": {
        "encoder_max_abs_error": TORCHSCRIPT_ENCODER_ATOL,
        "decoder_logit_max_abs_error": TORCHSCRIPT_DECODER_LOGIT_ATOL,
    },
    "onnx_is_preferred_but_not_required_for_phase7b_gate": True,
    "onnx_attempted": ATTEMPT_ONNX,
    "onnx_deferred_reason": (
        "legacy decoder export exceeded Kaggle 12-hour execution limit"
    ),
    "rows": export_rows,
    "synthetic_input_only": True,
    "validation_images_read": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})

print("PHASE 07B CELL 4 — EXPORT")
print("Status                  : PASSED")
for row in export_rows:
    print(
        f"{row['pair']:7s}                 : TorchScript PASSED | "
        f"ONNX {row['onnx_status']}"
    )
print("Input data              : SYNTHETIC ONLY")
print("Held-out Test           : SEALED")
print("Progress                : 4 / 5 code cells")


Exporting T1-S1: TorchScript encoder/decoder...
T1-S1 export complete: TorchScript PASSED; ONNX deferred_legacy_export_timeout_torchscript_fallback
Exporting T2-S2: TorchScript encoder/decoder...


/usr/local/lib/python3.13/dist-packages/timm/models/mobilevit.py:248: TracerWarning: Converting a tensor to a Python float might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  new_h, new_w = math.ceil(H / patch_h) * patch_h, math.ceil(W / patch_w) * patch_w
/usr/local/lib/python3.13/dist-packages/timm/models/mobilevit.py:252: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if new_h != H or new_w != W:


T2-S2 export complete: TorchScript PASSED; ONNX deferred_legacy_export_timeout_torchscript_fallback
Exporting T3-S3: TorchScript encoder/decoder...


/usr/local/lib/python3.13/dist-packages/timm/models/tiny_vit.py:390: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if H == self.window_size and W == self.window_size:
/usr/local/lib/python3.13/dist-packages/timm/models/tiny_vit.py:397: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  padding = pad_b > 0 or pad_r > 0
/usr/local/lib/python3.13/dist-packages/timm/models/tiny_vit.py:398: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as

T3-S3 export complete: TorchScript PASSED; ONNX deferred_legacy_export_timeout_torchscript_fallback
PHASE 07B CELL 4 — EXPORT
Status                  : PASSED
T1-S1                   : TorchScript PASSED | ONNX deferred_legacy_export_timeout_torchscript_fallback
T2-S2                   : TorchScript PASSED | ONNX deferred_legacy_export_timeout_torchscript_fallback
T3-S3                   : TorchScript PASSED | ONNX deferred_legacy_export_timeout_torchscript_fallback
Input data              : SYNTHETIC ONLY
Held-out Test           : SEALED
Progress                : 4 / 5 code cells


## Cell 5 — Đóng Phase 07B và cấp quyền Phase 07C


In [5]:
STARTED = time.perf_counter()
authorization = {
    "schema_version": "phase7c_export_parity_validation_authorized_v1",
    "created_at_utc": now(),
    "status": "authorized",
    "required_next": (
        "Phase 07C deterministic Validation-only export parity and CPU reference benchmark"
    ),
    "candidates": [
        {
            "pair": row["pair"],
            "torchscript_status": row["torchscript_status"],
            "onnx_status": row["onnx_status"],
        }
        for row in export_rows
    ],
    "constraints": {
        "new_training": False,
        "checkpoint_reselection": False,
        "fp32_only": True,
        "quantization": False,
        "validation_calibration_images": 100,
        "held_out_test_opening": False,
    },
    "test_set_status": "sealed",
    "test_set_used": False,
}
write_json(OUT / "PHASE7C_EXPORT_PARITY_VALIDATION_AUTHORIZED.json", authorization)

completion = {
    "schema_version": "phase7b_portable_model_factory_export_complete_v1",
    "created_at_utc": now(),
    "status": "complete",
    "decision": "portable_factory_and_fp32_export_passed_phase7c_authorized",
    "strict_checkpoint_loads": "3 / 3",
    "exact_parameter_contracts": "3 / 3",
    "torchscript_exports": "3 / 3",
    "onnx_results": {
        row["pair"]: row["onnx_status"] for row in export_rows
    },
    "new_training_performed": False,
    "validation_images_read": False,
    "test_set_status": "sealed",
    "test_set_used": False,
}

manifest_rows = []
for directory in [AUDIT, CONFIG, RESULT, EXPORT, PACKAGE]:
    for path in sorted(directory.rglob("*")):
        if path.is_file():
            manifest_rows.append({
                "relative_path": path.relative_to(OUT).as_posix(),
                "size_bytes": path.stat().st_size,
                "sha256": sha256(path),
            })
manifest_csv = MANIFEST / "phase7b_manifest.csv"
with manifest_csv.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=["relative_path", "size_bytes", "sha256"],
    )
    writer.writeheader()
    writer.writerows(manifest_rows)
manifest_json = MANIFEST / "phase7b_manifest.json"
write_json(manifest_json, {
    "schema_version": "phase7b_manifest_v1",
    "created_at_utc": now(),
    "rows": manifest_rows,
})
sha_file = MANIFEST / "SHA256SUMS"
sha_file.write_text(
    "".join(f"{row['sha256']}  {row['relative_path']}\n" for row in manifest_rows),
    encoding="utf-8",
)
completion["manifest"] = {
    "csv": {
        "relative_path": manifest_csv.relative_to(OUT).as_posix(),
        "sha256": sha256(manifest_csv),
        "rows": len(manifest_rows),
    },
    "json": {
        "relative_path": manifest_json.relative_to(OUT).as_posix(),
        "sha256": sha256(manifest_json),
    },
    "sha256sums": {
        "relative_path": sha_file.relative_to(OUT).as_posix(),
        "sha256": sha256(sha_file),
    },
}
completion["manifest_rows"] = len(manifest_rows)
completion["elapsed_seconds_cell05"] = time.perf_counter() - STARTED
write_json(OUT / "PHASE7B_PORTABLE_MODEL_FACTORY_EXPORT_COMPLETE.json", completion)

print("PHASE 07B — FINAL")
print("Status                  : COMPLETE")
print("Strict checkpoint load  : 3 / 3")
print("Parameter contracts     : 3 / 3")
print("TorchScript export      : 3 / 3")
print("New training            : NOT PERFORMED")
print("Validation images       : NOT READ")
print("Held-out Test           : SEALED / NOT USED")
print("Phase 07C               : AUTHORIZED")
print("Manifest                :", len(manifest_rows), "/", len(manifest_rows), "VERIFIED")
print("Progress                : 5 / 5 code cells")


PHASE 07B — FINAL
Status                  : COMPLETE
Strict checkpoint load  : 3 / 3
Parameter contracts     : 3 / 3
TorchScript export      : 3 / 3
New training            : NOT PERFORMED
Validation images       : NOT READ
Held-out Test           : SEALED / NOT USED
Phase 07C               : AUTHORIZED
Manifest                : 15 / 15 VERIFIED
Progress                : 5 / 5 code cells
